In [2]:
import sys

sys.path.append("/Users/liu.9639/Downloads/A2")

import torch
import matplotlib.pyplot as plt
from a2.models.transformer_seq2seq import *
from a2.data import *
# Import established script functions and configs from your repository
from train_transformer import *

from train_mlp import *


In [3]:
import argparse

# Define the parser
parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=10)
parser.add_argument("--batch-size", type=int, default=128)
parser.add_argument("--d-model", type=int, default=128)
parser.add_argument("--nhead", type=int, default=4)  # ensure d_model % nhead == 0
parser.add_argument("--num-layers", type=int, default=2)
parser.add_argument("--ffn-dim", type=int, default=256)
parser.add_argument(
    "--dropout",
    type=float,
    default=0.1,
    help="Ignored by simplified model (no dropout implemented); kept for CLI compatibility",
)
parser.add_argument("--lr", type=float, default=1e-3)
parser.add_argument("--max-digits", type=int, default=3)
parser.add_argument("--test-max-digits", type=int, default=None, help="Max digits for the test set (default: --max-digits); set larger to test generalization to longer numbers")
parser.add_argument("--train-size", type=int, default=20000)
parser.add_argument("--val-size", type=int, default=2000)
parser.add_argument("--test-size", type=int, default=2000)
parser.add_argument("--weight-decay", type=float, default=0.01, help="AdamW weight decay")
parser.add_argument("--seed", type=int, default=42, help="Seed for model init / batch order")
parser.add_argument("--data-seed", type=int, default=0, help="Seed for the train/val/test data (and the label permutation)")
parser.add_argument("--random-labels", action="store_true", help="Shuffle targets across the training set (Zhang et al. 2017)")

args_list = [
    "--epochs", "10",
    "--max-digits", "3",
    "--train-size", "20000",
    "--val-size", "2000",
    "--test-size", "2000",
]


# Parse the argument list
args = parser.parse_args(args_list)

# Access your arguments as usual
print(args)

Namespace(epochs=10, batch_size=128, d_model=128, nhead=4, num_layers=2, ffn_dim=256, dropout=0.1, lr=0.001, max_digits=3, test_max_digits=None, train_size=20000, val_size=2000, test_size=2000, weight_decay=0.01, seed=42, data_seed=0, random_labels=False)


In [4]:

set_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = default_vocab()

train_ex = generate_dataset(args.train_size, args.max_digits, seed=args.data_seed)
val_ex = generate_dataset(args.val_size, args.max_digits, seed=args.data_seed + 1)
test_max_digits = args.test_max_digits or args.max_digits
test_ex = generate_dataset(args.test_size, test_max_digits, seed=args.data_seed + 2)
if args.random_labels:
    train_ex = permute_targets(train_ex, seed=args.data_seed)
    print("Training targets randomly permuted (--random-labels); val/test are unchanged.")

train_ds = AdditionDataset(train_ex, vocab)
val_ds = AdditionDataset(val_ex, vocab)
test_ds = AdditionDataset(test_ex, vocab)

collate = lambda b: collate_fn(b, pad_id=vocab.pad_id)  # noqa: E731
train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate)
train_eval_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)

print(f"Datasets: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}")

model = TransformerSeq2Seq(
    TransformerConfig(
        vocab_size=vocab.size,
        d_model=args.d_model,
        nhead=args.nhead,
        num_layers=args.num_layers,
        dim_feedforward=args.ffn_dim,
        dropout=args.dropout,
        max_len=128,
    ),
    pad_id=vocab.pad_id,
    sos_id=vocab.sos_id,
    eos_id=vocab.eos_id,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

best_val = 0.0
for epoch in range(1, args.epochs + 1):
    print(f"\nEpoch {epoch}/{args.epochs}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    # train_acc above is a running average over minibatches *during* the epoch; re-evaluate
    # the final weights on the whole training set for a clean train exact-match number.
    _, train_em = evaluate(model, train_eval_loader, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} (full-set EM {train_em:.4f}) | val loss {val_loss:.4f} acc {val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test  | loss {test_loss:.4f} acc {test_acc:.4f}")



Datasets: train=20000 val=2000 test=2000
Batches: train=157 val=16 test=16

Epoch 1/10


Epoch 01 | train loss 7.3911 acc 0.0005 (full-set EM 0.0012) | val loss 1.7866 acc 0.0010

Epoch 2/10


Epoch 02 | train loss 1.7306 acc 0.0011 (full-set EM 0.0017) | val loss 1.7128 acc 0.0020

Epoch 3/10


Epoch 03 | train loss 1.6591 acc 0.0017 (full-set EM 0.0019) | val loss 1.6188 acc 0.0015

Epoch 4/10


Epoch 04 | train loss 1.5624 acc 0.0030 (full-set EM 0.0038) | val loss 1.5275 acc 0.0010

Epoch 5/10


Epoch 05 | train loss 1.4268 acc 0.0053 (full-set EM 0.0077) | val loss 1.3466 acc 0.0083

Epoch 6/10


... (2 outputs left out)

Epoch 08 | train loss 1.1280 acc 0.0330 (full-set EM 0.0644) | val loss 0.9939 acc 0.0563

Epoch 9/10


Epoch 09 | train loss 0.8505 acc 0.0931 (full-set EM 0.1507) | val loss 0.7360 acc 0.1241

Epoch 10/10


Epoch 10 | train loss 0.5614 acc 0.2958 (full-set EM 0.4222) | val loss 0.4433 acc 0.4095


Test  | loss 0.4437 acc 0.3998
